# Totals Model — Walk-Forward CV + Production Training

Trains XGBoost and Ridge regressors to predict `total_points - total_line`
(deviation from the Vegas total). Saves production pkls to `betting/models/`.

**Strategy:** the target is the *residual* from Vegas, not the raw total.
Vegas's own `total_line` is included as a feature, so the model learns small
adjustments on top of it. Hit-rate is measured as: sign(predicted residual)
== sign(actual residual).

**Tier logic (conservative v1):**
- `HIGH` = both XGBoost and Ridge predict UNDER (both residuals < 0)
- `PASS` = any other combination (including all OVERs — no OVER edge found in CV)

Walk-forward CV results (6 folds, 2020-2025, n=1,675 push-excluded):
- XGBoost: 52.3% ± 1.9%
- Ridge: 52.1% ± 2.5%
- RF: 53.3% ± 2.1% (best single model)
- Consensus UNDER (XGB + Ridge): 55.7% on n=575 — 95% CI 51.6-59.7%

## Section 1 — Configuration

In [ ]:
from pathlib import Path

TRAIN_SEASONS = list(range(2014, 2025))  # 2014-2024 for production pkls
TEST_SEASONS  = [2025]                    # live holdout
COACH_SEASONS = list(range(1999, 2026))
ALL_SEASONS   = TRAIN_SEASONS + TEST_SEASONS

# Resolve paths (works whether CWD is project root or betting/)
ALLPRO_CSV = next(
    (p for p in [Path('nfl_allpro_1997_2025.csv'), Path('betting/nfl_allpro_1997_2025.csv')]
     if p.exists()),
    Path('betting/nfl_allpro_1997_2025.csv')
)
WEATHER_CSV = next(
    (p for p in [Path('nfl_weather_2014_2025.csv'), Path('betting/nfl_weather_2014_2025.csv')]
     if p.exists()),
    Path('betting/nfl_weather_2014_2025.csv')
)
MODELS_DIR = ALLPRO_CSV.parent / 'models'
MODELS_DIR.mkdir(exist_ok=True)
print(f'AllPro CSV: {ALLPRO_CSV}  exists={ALLPRO_CSV.exists()}')
print(f'Weather CSV: {WEATHER_CSV}  exists={WEATHER_CSV.exists()}')
print(f'Models dir: {MODELS_DIR}')


In [ ]:
# ── Section 1 tests ──
assert ALLPRO_CSV.exists(), f'AllPro CSV not found at {ALLPRO_CSV}'
assert WEATHER_CSV.exists(), f'Weather CSV not found at {WEATHER_CSV} — run betting/experiments/fetch_weather.py'
print('Section 1 tests passed')


## Section 2 — Imports

In [ ]:
import warnings, subprocess, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
import nflreadpy as nfl
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import joblib

# Load totals_features.ipynb into this namespace via json+exec
import json as _json
_TF_CANDIDATES = [
    Path('totals_features.ipynb'),
    Path('betting/totals_features.ipynb'),
]
_tf_path = next((p for p in _TF_CANDIDATES if p.exists()), None)
if _tf_path is None:
    raise FileNotFoundError(f'totals_features.ipynb not found; tried: {_TF_CANDIDATES}')
RUN_TESTS = False
with open(_tf_path, encoding='utf-8') as _fh:
    _tf_nb = _json.load(_fh)
for _cell in _tf_nb['cells']:
    if _cell['cell_type'] == 'code':
        exec(''.join(_cell['source']), globals())
globals().pop('_tf_path', None); globals().pop('_tf_nb', None); globals().pop('_fh', None); globals().pop('_cell', None)
print(f'Loaded totals_features.ipynb — TOTALS_FEATURE_COLS={len(TOTALS_FEATURE_COLS)}')

# Also load spread features.ipynb for the base 35-feature set
_SF_CANDIDATES = [
    Path('features.ipynb'),
    Path('betting/features.ipynb'),
]
_sf_path = next((p for p in _SF_CANDIDATES if p.exists()), None)
if _sf_path is None:
    raise FileNotFoundError(f'features.ipynb not found; tried: {_SF_CANDIDATES}')
RUN_TESTS = False
with open(_sf_path, encoding='utf-8') as _fh:
    _sf_nb = _json.load(_fh)
for _cell in _sf_nb['cells']:
    if _cell['cell_type'] == 'code':
        exec(''.join(_cell['source']), globals())
globals().pop('_sf_path', None); globals().pop('_sf_nb', None); globals().pop('_fh', None); globals().pop('_cell', None)
print(f'Loaded features.ipynb — PROD_FEATURES_35={len(PROD_FEATURES_35)}')


In [ ]:
# ── Section 2 tests ──
assert len(TOTALS_FEATURE_COLS) == 14, f'Expected 14 totals cols, got {len(TOTALS_FEATURE_COLS)}'
assert len(PROD_FEATURES_35) == 35, f'Expected 35 spread cols, got {len(PROD_FEATURES_35)}'
overlap = set(TOTALS_FEATURE_COLS) & set(PROD_FEATURES_35)
assert not overlap, f'Totals and spread features overlap: {overlap}'
print('Section 2 tests passed | no feature overlap between spread and totals')


## Section 3 — Data Loading

Reuses the spread pipeline's schedule + PBP load. The `sched` and `pbp_full` objects are the same ones used by `model_comparison.ipynb` — no extra downloading.

In [ ]:
print(f'Loading schedules {min(COACH_SEASONS)}-{max(COACH_SEASONS)}...')
sched_full = nfl.load_schedules(COACH_SEASONS).to_pandas()
sched = sched_full[sched_full['season'].isin(ALL_SEASONS)].copy()
sched = sched[sched['result'].notna()].copy()
print(f'  completed games: {len(sched):,}')

print(f'Loading PBP {min(ALL_SEASONS)}-{max(ALL_SEASONS)}...')
pbp_full = nfl.load_pbp(ALL_SEASONS).to_pandas()
pbp_rp = pbp_full[
    pbp_full['play_type'].isin(['run', 'pass']) &
    pbp_full['posteam'].notna() & pbp_full['defteam'].notna()
].copy()
print(f'  run/pass plays: {len(pbp_rp):,}')


In [ ]:
# ── Section 3 tests ──
assert not sched.empty and sched['result'].notna().all()
assert len(pbp_rp) > 300_000, f'PBP unexpectedly small: {len(pbp_rp)}'
print('Section 3 tests passed')


## Section 4 — Build Spread Feature Matrix

Loads the production 85-feature spread pipeline from `model_comparison.ipynb` cells 1-37 via the same json+exec approach used by `tune_time_decay.py`. This populates `g` with the full 120-column games table and `avail` with the 35-feature production subset.

In [ ]:
import sys as _sys
_sys.path.insert(0, str(ALLPRO_CSV.parent / 'experiments'))
from tune_time_decay import load_prep_cells as _load_prep

_ns = _load_prep(earliest=2014)
g       = _ns['g']
avail   = _ns['avail']   # 35 spread features
print(f'Spread features: g.shape={g.shape}  avail={len(avail)}')


In [ ]:
# ── Section 4 tests ──
assert len(avail) == 35, f'Expected 35 spread features, got {len(avail)}'
assert 'total_line' in g.columns, 'total_line missing from g'
assert 'spread_line' in g.columns, 'spread_line missing from g'
print('Section 4 tests passed')


## Section 5 — Build Totals Feature Matrix

In [ ]:
g = build_totals_features(g, sched, pbp_full, weather_path=WEATHER_CSV)
g['total_points'] = g['home_score'] + g['away_score']
g = g[g['total_line'].notna() & g['total_points'].notna()].copy()

TOTALS_ALL_COLS = list(avail) + TOTALS_FEATURE_COLS  # 35 + 14 = 49
g['total_diff'] = g['total_points'] - g['total_line']  # target

print(f'Games after filter: {len(g):,}')
print(f'Total features: {len(TOTALS_ALL_COLS)} (35 spread + 14 totals)')
print(f'is_dome games: {g["is_dome"].sum():,} ({g["is_dome"].mean()*100:.1f}%)')
print(f'Total diff: mean={g["total_diff"].mean():.2f}  OVER={( g["total_diff"]>0).mean()*100:.1f}%  UNDER={(g["total_diff"]<0).mean()*100:.1f}%')


In [ ]:
# ── Section 5 tests ──
assert len(TOTALS_ALL_COLS) == 49, f'Expected 49 total features, got {len(TOTALS_ALL_COLS)}'
for _c in TOTALS_FEATURE_COLS:
    assert _c in g.columns, f'Missing totals feature: {_c}'
    assert g[_c].notna().all(), f'NaN in {_c}'
assert g['is_dome'].sum() > 100, 'Expected >100 dome games'
_alg = (g['home_implied_pts'] + g['away_implied_pts'] - g['total_line']).abs().max()
assert _alg < 1e-6, f'Implied total algebra error: {_alg}'
print(f'Section 5 tests passed | {len(g):,} games, {len(TOTALS_ALL_COLS)} features, no nulls')


## Section 6 — Walk-Forward CV (6 folds, 2020-2025)

Validates the totals models before retraining on full data. Tests on each year 2020-2025 with training on all prior years from 2014 onward.

In [ ]:
def _make_fold_data(train_seasons, test_season):
    tr_m = g['season'].isin(train_seasons)
    te_m = g['season'] == test_season
    X_tr = g.loc[tr_m, TOTALS_ALL_COLS].fillna(0).values.astype('float32')
    y_tr = g.loc[tr_m, 'total_diff'].values.astype('float32')
    X_te = g.loc[te_m, TOTALS_ALL_COLS].fillna(0).values.astype('float32')
    y_te = g.loc[te_m, 'total_diff'].values.astype('float32')
    total_line_te   = g.loc[te_m, 'total_line'].values
    actual_total_te = g.loc[te_m, 'total_points'].values
    return X_tr, y_tr, X_te, y_te, total_line_te, actual_total_te

cv_results = {}
for model_name in ['XGBoost', 'Ridge']:
    accs = []
    for test_yr in range(2020, 2026):
        train_yrs = list(range(2014, test_yr))
        X_tr, y_tr, X_te, y_te, tl_te, at_te = _make_fold_data(train_yrs, test_yr)
        if model_name == 'XGBoost':
            m = xgb.XGBRegressor(
                n_estimators=500, max_depth=3, learning_rate=0.01, min_child_weight=3,
                subsample=0.6, colsample_bytree=0.6, reg_alpha=2.0, reg_lambda=5.0,
                objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0)
            m.fit(X_tr, y_tr)
            preds_diff = m.predict(X_te)
        else:
            sc = StandardScaler()
            m  = Ridge(alpha=50.0)
            m.fit(sc.fit_transform(X_tr), y_tr)
            preds_diff = m.predict(sc.transform(X_te))
        preds_total = tl_te + preds_diff
        accs.append(totals_acc(preds_total, tl_te, at_te))
    cv_results[model_name] = accs
    mean_acc = float(np.mean(accs))
    std_acc  = float(np.std(accs))
    marker = 'ABOVE break-even' if mean_acc > 0.524 else 'below break-even'
    print(f'  {model_name:8}  CV: {mean_acc:.1%} +/- {std_acc:.1%}  [{marker}]')

# Consensus UNDER
under_correct, under_total = 0, 0
for test_yr in range(2020, 2026):
    train_yrs = list(range(2014, test_yr))
    X_tr, y_tr, X_te, y_te, tl_te, at_te = _make_fold_data(train_yrs, test_yr)
    sc  = StandardScaler()
    r_m = Ridge(alpha=50.0);  r_m.fit(sc.fit_transform(X_tr), y_tr)
    x_m = xgb.XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.01,
                            min_child_weight=3, subsample=0.6, colsample_bytree=0.6,
                            reg_alpha=2.0, reg_lambda=5.0, objective='reg:squarederror',
                            random_state=42, n_jobs=-1, verbosity=0)
    x_m.fit(X_tr, y_tr)
    ridge_pred = r_m.predict(sc.transform(X_te))
    xgb_pred   = x_m.predict(X_te)
    push = at_te == tl_te
    both_under = (ridge_pred < 0) & (xgb_pred < 0) & ~push
    under_correct += ((at_te[both_under] < tl_te[both_under])).sum()
    under_total   += both_under.sum()

print(f'  Consensus UNDER (XGB+Ridge agree): {under_correct/under_total:.1%} on n={under_total} picks')


In [ ]:
# ── Section 6 tests ──
for _name, _accs in cv_results.items():
    assert len(_accs) == 6, f'{_name} expected 6 folds'
    assert all(0.35 <= a <= 0.70 for a in _accs), f'{_name} fold ATS out of range: {_accs}'
    assert float(np.mean(_accs)) > 0.48, f'{_name} CV mean suspiciously low: {np.mean(_accs):.3f}'
print('Section 6 tests passed')


## Section 7 — Production Retrain + Save PKLs

Retrains on ALL finished seasons (2014-2024) and saves `totals_xgboost.pkl` and `totals_ridge.pkl` to `betting/models/`.

**WARNING:** This cell overwrites production pkls.

In [ ]:
full_tr_m = g['season'].isin(list(range(2014, 2025)))
X_prod = g.loc[full_tr_m, TOTALS_ALL_COLS].fillna(0).values.astype('float32')
y_prod = g.loc[full_tr_m, 'total_diff'].values.astype('float32')
print(f'Production training: {len(X_prod):,} games  features: {len(TOTALS_ALL_COLS)}')

# XGBoost
xgb_totals = xgb.XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.01, min_child_weight=3,
    subsample=0.6, colsample_bytree=0.6, reg_alpha=2.0, reg_lambda=5.0,
    objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0)
xgb_totals.fit(X_prod, y_prod)
print('XGBoost retrained.')

# Ridge
scaler_totals  = StandardScaler()
ridge_totals   = Ridge(alpha=50.0)
ridge_totals.fit(scaler_totals.fit_transform(X_prod), y_prod)
print('Ridge retrained.')

# Save
joblib.dump(
    {'model': xgb_totals, 'feature_cols': TOTALS_ALL_COLS,
     'target': 'total_diff', 'train_seasons': list(range(2014, 2025))},
    str(MODELS_DIR / 'totals_xgboost.pkl'))
print(f'Saved -> totals_xgboost.pkl')

joblib.dump(
    {'model': ridge_totals, 'scaler': scaler_totals, 'feature_cols': TOTALS_ALL_COLS,
     'target': 'total_diff', 'train_seasons': list(range(2014, 2025))},
    str(MODELS_DIR / 'totals_ridge.pkl'))
print(f'Saved -> totals_ridge.pkl')
print('Production pkls saved.')


In [ ]:
# ── Section 7 tests ──
import joblib as _jl
for _name, _expected_keys in [
    ('totals_xgboost.pkl', {'model', 'feature_cols', 'target'}),
    ('totals_ridge.pkl',   {'model', 'scaler', 'feature_cols', 'target'}),
]:
    _p = MODELS_DIR / _name
    assert _p.exists(), f'{_name} not written'
    assert _p.stat().st_size > 1_000, f'{_name} unexpectedly small'
    _loaded = _jl.load(str(_p))
    assert _expected_keys <= set(_loaded.keys()), f'{_name} missing keys: {_expected_keys - set(_loaded.keys())}'
    assert len(_loaded['feature_cols']) == 49, f'{_name} feature_cols wrong length'
print('Section 7 tests passed | both pkl files written and verified')
